# Benchmark de corrección OCR. Manualito

Mide el pipeline de corrección de la [Issue #9](https://gitlab.com/HP-SCDS/Observatorio/2025-2026/manualito/ubu-manualito/-/issues/9) (gate de confianza, reglas
deterministas y consenso LLM) sobre un dataset versionado de líneas OCR reales
y baterías sintéticas, con resultados reproducibles e idempotentes.

## 1. Objetivo

Cuantificar cuántas líneas rotas quedan exactamente corregidas, cuántas líneas
sanas se dañan (sobrecorrección, restricción dura del diseño) y cómo evoluciona
la tasa de error de carácter (CER), separando la aportación de las reglas
deterministas de la de la pasada LLM por consenso.

## 2. Metodología

Dos bloques complementarios:

- **Flujo de producto** sobre las líneas `align_*` (OCR real de tres manuales
  con verdad terreno transcrita a mano): se replica el comportamiento del
  gateway. Gate de tres zonas por confianza, reglas deterministas sobre las
  supervivientes y pasada LLM solo en la franja media.
- **Corrector aislado** sobre las baterías sintéticas (`typos`, `diacriticos`,
  `segmentacion`): cada línea se envía siempre al endpoint `/correct-line`
  para medir la destreza del consenso sin el gate delante.

La fase LLM ejecuta el servicio real (`manualito-llm`) vía `docker exec`, con
el modelo corrector configurado. Las métricas se calculan con la misma
normalización y Levenshtein que las campañas experimentales previas. El diseño
del pipeline y sus umbrales provienen de evidencia experimental cerrada (véanse las referencias al final).

Limitación declarada: el des-guionado de fin de línea opera entre líneas
consecutivas de una página y este dataset evalúa líneas aisladas, así que esa
regla queda fuera de ambos bloques (su tasa 7/8 con guarda se midió aparte en
la campaña de operaciones).

## 3. Configuración

`RUN_BENCHMARK` controla el modo lectura (por defecto no se ejecuta nada
pesado). `FORCE` reemplaza una medición existente con la misma configuración.
`FLOW_VERSION` versiona el flujo medido: cambiarlo genera una medición nueva
sin tocar las anteriores.

In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
sys.dont_write_bytecode = True
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

RUN_BENCHMARK = False
FORCE = False

FLOW_VERSION = 1
CONTAINER_NAME = "manualito-llm"
CORRECTION_ENDPOINT = "http://localhost:8003/correct-line"
METRICS = ["exactas", "sobrecorreccion", "cer"]
YELLOW = "\033[33m"
CYAN = "\033[36m"
GREEN = "\033[32m"
RED = "\033[31m"
RESET = "\033[0m"

def find_benchmark_dir(start: Path) -> Path:
    for base in (start.resolve(), *start.resolve().parents):
        local = base / "dataset" / "casos.json"
        nested = base / "docs" / "benchmarks" / "ocr" / "correction" / "dataset" / "casos.json"
        if local.is_file():
            return base
        if nested.is_file():
            return nested.parent.parent
    raise RuntimeError("No se encontró el benchmark de corrección OCR")

NOTEBOOK_DIR = find_benchmark_dir(Path.cwd())
REPO_ROOT = next(
    (candidate for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents) if (candidate / "pyproject.toml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("No se encontró la raíz del repositorio")
DATASET_PATH = NOTEBOOK_DIR / "dataset" / "casos.json"
MANIFEST_PATH = NOTEBOOK_DIR / "dataset" / "manifest.json"
OUTPUTS_DIR = NOTEBOOK_DIR
RESULTS_PATH = OUTPUTS_DIR / "results.json"


sys.path.insert(0, str(REPO_ROOT / "backend"))

from common.ocr.correction_rules import (  # noqa: E402
    OcrCorrectionConfig,
    collapse_guide_dots,
    strip_bullet_artifact,
)

DISCARD_BELOW = 0.5
LLM_BELOW = 0.85
CORRECTION_CONFIG = OcrCorrectionConfig(discard_below=DISCARD_BELOW, llm_below=LLM_BELOW)


def info(text: str, highlight: object = "") -> None:
    message = text.rstrip(". ")
    if highlight != "":
        message += f" {GREEN}{highlight}{CYAN}"
    print(f"{YELLOW}[*]{RESET} {CYAN}{message}.{RESET}")


def abort(message: str) -> None:
    """Detiene el notebook con un mensaje claro."""
    print(f"{RED}[!] ERROR: {message.rstrip('. ')}.{RESET}", file=sys.stderr)
    raise RuntimeError(message)


def git_revision() -> str | None:
    """Devuelve la revisión git corta o None si no está disponible."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
            cwd=REPO_ROOT,
        )
    except (OSError, subprocess.CalledProcessError):
        return None
    return result.stdout.strip() or None


def write_utf8(path: Path, content: str) -> None:
    """Escribe texto UTF-8 con saltos de línea LF."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.write(content)


def format_metric(value: float) -> str:
    """Formatea una métrica con cuatro decimales y coma decimal."""
    return f"{value:.4f}".replace(".", ",")


info("Configuración", f"RUN_BENCHMARK={RUN_BENCHMARK}, FORCE={FORCE}, FLOW_VERSION={FLOW_VERSION}")

[*] Configuración RUN_BENCHMARK=False, FORCE=False, FLOW_VERSION=1.


## 4. Dataset versionado

73 líneas con verdad terreno: 37 reales de tres manuales escaneados (con
confianza OCR y ventana de contexto ±2) y 36 sintéticas de las baterías de
typos, diacríticos y segmentación. El manifiesto fija el hash canónico del
dataset y cualquier medición lo referencia.

In [2]:
def canonical_json_sha256(data: Any) -> str:
    """Hash canónico independiente de sangría y de finales de línea."""
    canonical = json.dumps(data, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def normalize_text(text: str) -> str:
    """Normaliza como las campañas: colapsa puntos de guía y espacios."""
    import re

    collapsed = re.sub(r"\.{3,}", " ", text)
    return re.sub(r"\s+", " ", collapsed).strip()


def levenshtein(first: str, second: str) -> int:
    """Distancia de edición clásica entre dos cadenas."""
    if len(first) < len(second):
        first, second = second, first
    previous = list(range(len(second) + 1))
    for row, char_a in enumerate(first, 1):
        current = [row]
        for col, char_b in enumerate(second, 1):
            current.append(
                min(previous[col] + 1, current[col - 1] + 1, previous[col - 1] + (char_a != char_b))
            )
        previous = current
    return previous[-1]


def cer(candidate: str, expected: str) -> float:
    """Tasa de error de carácter sobre el texto esperado normalizado."""
    normal_candidate = normalize_text(candidate)
    normal_expected = normalize_text(expected)
    return levenshtein(normal_candidate, normal_expected) / max(len(normal_expected), 1)


dataset_bundle = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
CASES = dataset_bundle["casos"]
DATASET_SHA256 = canonical_json_sha256(dataset_bundle)

if len(CASES) != 73:
    abort(f"dataset inesperado: {len(CASES)} casos")
if len({case["id"] for case in CASES}) != len(CASES):
    abort("ids duplicados en el dataset")
for case in CASES:
    if set(case) != {"id", "texto_ocr", "confianza", "contexto_antes", "contexto_despues", "esperado", "categoria"}:
        abort(f"campos inesperados en {case.get('id')}")

if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if manifest["sha256_canonico"] != DATASET_SHA256:
        abort("el manifiesto no coincide con el dataset")

PRODUCT_CASES = [case for case in CASES if case["categoria"].startswith("align_")]
BATTERY_CASES = [case for case in CASES if not case["categoria"].startswith("align_")]
SANE_IDS = {
    case["id"]
    for case in CASES
    if normalize_text(case["texto_ocr"]) == normalize_text(case["esperado"])
}
info("Dataset verificado", f"sha256={DATASET_SHA256[:12]}, producto={len(PRODUCT_CASES)}, baterías={len(BATTERY_CASES)}, sanas={len(SANE_IDS)}")

[*] Dataset verificado sha256=fd730bedd93d, producto=37, baterías=36, sanas=14.


## 5. Acceso al flujo real

La fase de reglas se ejecuta en local importando `backend/common/ocr` (los
mismos módulos que usa el gateway). La fase LLM entra por el contrato público
del servicio: un script dentro del contenedor `manualito-llm` envía cada línea
a `POST /correct-line` y devuelve las salidas. El servicio debe estar levantado
con `OLLAMA_CORRECTION_MODEL` configurado (perfil `high`).

In [3]:
CONTAINER_SCRIPT = r"""
import json
import sys

import httpx

requests = json.loads(sys.stdin.read())
results = {}
with httpx.Client(timeout=600) as client:
    for item in requests:
        try:
            response = client.post(item["url"], json=item["payload"])
            response.raise_for_status()
            results[item["id"]] = {"ok": True, "text": response.json()["text"]}
        except Exception as error:
            results[item["id"]] = {"ok": False, "error": str(error)}
        print(f"caso {item['id']} listo", file=sys.stderr, flush=True)
print(json.dumps(results, ensure_ascii=False))
"""


def apply_rules_stage(case: dict[str, Any]) -> dict[str, Any]:
    """Replica gate y reglas por línea del flujo de producto."""
    confidence = case["confianza"]
    if confidence is not None and confidence < DISCARD_BELOW:
        return {"discarded": True, "text": ""}
    cleaned = collapse_guide_dots(strip_bullet_artifact(case["texto_ocr"])).strip()
    in_band = confidence is not None and DISCARD_BELOW <= confidence < LLM_BELOW
    return {"discarded": False, "text": cleaned, "llm_band": in_band}


def run_llm_stage(requests: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    """Ejecuta las peticiones de corrección dentro del contenedor llm."""
    if not requests:
        return {}
    payload = json.dumps(requests, ensure_ascii=False)
    completed = subprocess.run(
        ["docker", "exec", "-i", "-e", "PYTHONUTF8=1", CONTAINER_NAME, "python", "-c", CONTAINER_SCRIPT],
        input=payload.encode("utf-8"),
        capture_output=True,
        check=False,
    )
    if completed.returncode != 0:
        abort(f"fallo en el contenedor: {completed.stderr.decode('utf-8', 'replace')[-400:]}")
    stdout = completed.stdout.decode("utf-8")
    return json.loads(stdout[stdout.index("{"):])


def measure_pipeline() -> dict[str, dict[str, Any]]:
    """Ejecuta reglas y LLM y devuelve la salida final por caso."""
    stages = {case["id"]: apply_rules_stage(case) for case in CASES}
    requests = []
    for case in CASES:
        stage = stages[case["id"]]
        is_battery = not case["categoria"].startswith("align_")
        if stage["discarded"] or not (is_battery or stage.get("llm_band")):
            continue
        requests.append(
            {
                "id": case["id"],
                "url": CORRECTION_ENDPOINT,
                "payload": {
                    "text": stage["text"],
                    "context_before": case["contexto_antes"][-2:],
                    "context_after": case["contexto_despues"][:2],
                    "language": "es",
                },
            }
        )
    llm_results = run_llm_stage(requests)
    outputs = {}
    for case in CASES:
        stage = stages[case["id"]]
        llm_result = llm_results.get(case["id"])
        if llm_result is not None and not llm_result["ok"]:
            abort(f"fallo LLM en {case['id']}: {llm_result['error'][:200]}")
        final_text = llm_result["text"] if llm_result is not None else stage["text"]
        outputs[case["id"]] = {
            "descartada": stage["discarded"],
            "llm": llm_result is not None,
            "salida": "" if stage["discarded"] else final_text,
        }
    return outputs


info("Casos de producto en franja LLM", sum(1 for case in PRODUCT_CASES if apply_rules_stage(case).get("llm_band")))

[*] Casos de producto en franja LLM 20.


## 6. Resultados

Cada medición registra configuración, hash canónico, métricas por bloque y por
categoría y la salida de cada caso. Con la misma configuración se reutiliza la
medición existente salvo `FORCE`.

In [4]:
CONFIGURATION = {
    "dataset_sha256_canonico": DATASET_SHA256,
    "flujo_version": FLOW_VERSION,
    "umbral_descarte": DISCARD_BELOW,
    "umbral_franja_llm": LLM_BELOW,
    "contenedor": CONTAINER_NAME,
    "metricas": METRICS,
}
CONFIGURATION_HASH = canonical_json_sha256(CONFIGURATION)


def measure_block(cases: list[dict[str, Any]], outputs: dict[str, dict[str, Any]]) -> dict[str, Any]:
    """Calcula métricas de un bloque de casos a partir de las salidas."""
    broken = [case for case in cases if case["id"] not in SANE_IDS]
    sane = [case for case in cases if case["id"] in SANE_IDS]
    evaluated = [case for case in broken if not outputs[case["id"]]["descartada"]]
    exact = sum(
        1
        for case in evaluated
        if normalize_text(outputs[case["id"]]["salida"]) == normalize_text(case["esperado"])
    )
    touched_sane = sum(
        1
        for case in sane
        if not outputs[case["id"]]["descartada"]
        and normalize_text(outputs[case["id"]]["salida"]) != normalize_text(case["texto_ocr"])
    )
    cer_before = sum(cer(case["texto_ocr"], case["esperado"]) for case in evaluated)
    cer_after = sum(cer(outputs[case["id"]]["salida"], case["esperado"]) for case in evaluated)
    return {
        "rotas": len(broken),
        "evaluadas": len(evaluated),
        "descartadas": len(broken) - len(evaluated),
        "exactas": exact,
        "tasa_exactas": round(exact / max(len(evaluated), 1), 4),
        "sanas": len(sane),
        "sobrecorreccion": touched_sane,
        "cer_antes": round(cer_before / max(len(evaluated), 1), 4),
        "cer_despues": round(cer_after / max(len(evaluated), 1), 4),
    }


def baseline_outputs() -> dict[str, dict[str, Any]]:
    """Salidas de la línea base sin corrección alguna."""
    return {
        case["id"]: {"descartada": False, "llm": False, "salida": case["texto_ocr"]}
        for case in CASES
    }


def build_measurement(outputs: dict[str, dict[str, Any]], *, kind: str) -> dict[str, Any]:
    """Construye una medición completa con métricas por bloque y categoría."""
    categories = sorted({case["categoria"] for case in CASES})
    by_category = {
        category: measure_block(
            [case for case in CASES if case["categoria"] == category], outputs
        )
        for category in categories
    }
    return {
        "id": f"local-{CONFIGURATION_HASH[:12]}" if kind == "medicion_local" else "linea-base",
        "tipo": kind,
        "fecha": datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z"),
        "dataset_sha256_canonico": DATASET_SHA256,
        "configuracion_hash": CONFIGURATION_HASH if kind == "medicion_local" else None,
        "revision_git": git_revision(),
        "configuracion": CONFIGURATION if kind == "medicion_local" else None,
        "metricas": {
            "producto": measure_block(PRODUCT_CASES, outputs),
            "corrector": measure_block(BATTERY_CASES, outputs),
        },
        "por_categoria": by_category,
        "casos": [
            {
                "id": case["id"],
                "salida": outputs[case["id"]]["salida"],
                "descartada": outputs[case["id"]]["descartada"],
                "llm": outputs[case["id"]]["llm"],
            }
            for case in CASES
        ],
    }


def validate_results(bundle: dict[str, Any]) -> None:
    """Aborta si el fichero de resultados viola el contrato."""
    if bundle.get("version") != 2:
        abort("results.json debe declarar version 2")
    measurements = bundle.get("mediciones", [])
    if not measurements:
        abort("results.json sin mediciones")
    if measurements[0]["tipo"] != "linea_base_sin_correccion":
        abort("la primera medición debe ser la línea base")
    identifiers = [measurement["id"] for measurement in measurements]
    if len(set(identifiers)) != len(identifiers):
        abort("ids de medición duplicados")
    for measurement in measurements:
        if measurement["dataset_sha256_canonico"] != DATASET_SHA256:
            abort("medición con dataset distinto al actual")
        for block in ("producto", "corrector"):
            metrics = measurement["metricas"][block]
            if metrics["rotas"] < metrics["exactas"]:
                abort("métricas incoherentes")
        expected_ids = [case["id"] for case in CASES]
        if [case["id"] for case in measurement["casos"]] != expected_ids:
            abort("casos desalineados con el dataset")


def load_results() -> dict[str, Any]:
    """Carga el fichero de resultados o crea la estructura inicial."""
    if RESULTS_PATH.exists():
        return json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
    return {"version": 2, "mediciones": [build_measurement(baseline_outputs(), kind="linea_base_sin_correccion")]}


def save_results(bundle: dict[str, Any]) -> None:
    """Valida y persiste resultados y documentos generados."""
    validate_results(bundle)
    write_utf8(RESULTS_PATH, json.dumps(bundle, ensure_ascii=False, indent=1) + "\n")

## Conclusiones

El corrector aislado arregla por completo 31 de 34 líneas con errores. En el
flujo de producto solo corrige 2 de las 24 evaluadas y apenas reduce el error
de caracteres, de 0,1849 a 0,1804. No daña ninguna línea que ya estuviera bien.

El buen resultado con errores sintéticos no se traslada al flujo completo.
La comparación usa las mismas 24 líneas, porque la medición descartó una de
las 25 que tenía la base.

In [5]:
results_bundle = load_results()

existing = next(
    (
        measurement
        for measurement in results_bundle["mediciones"]
        if measurement["tipo"] == "medicion_local"
        and measurement["configuracion_hash"] == CONFIGURATION_HASH
    ),
    None,
)

if RUN_BENCHMARK and (existing is None or FORCE):
    measurement = build_measurement(measure_pipeline(), kind="medicion_local")
    if existing is not None:
        index = results_bundle["mediciones"].index(existing)
        results_bundle["mediciones"][index] = measurement
    else:
        results_bundle["mediciones"].append(measurement)
    save_results(results_bundle)
    info("Medición guardada", measurement["id"])
elif existing is not None:
    info("Modo lectura: se reutiliza la medición persistida", existing["id"])
else:
    validate_results(results_bundle)
    info("Modo lectura: sin medición local para esta configuración")

for block in ("producto", "corrector"):
    metrics = results_bundle["mediciones"][-1]["metricas"][block]
    info(
        f"Resumen {block}",
        f"exactas={metrics['exactas']}/{metrics['evaluadas']}, sobrecorrección={metrics['sobrecorreccion']}, CER={format_metric(metrics['cer_despues'])}",
    )

[*] Modo lectura: se reutiliza la medición persistida local-515b643f689b.
[*] Resumen producto exactas=2/24, sobrecorrección=0, CER=0,1804.
[*] Resumen corrector exactas=31/34, sobrecorrección=0, CER=0,0037.


## Repetir la medición

Por defecto se valida el contrato y se muestra el resumen sin consultar servicios.

```bash
uv run --locked --only-group benchmarks jupyter execute docs/benchmarks/ocr/correction/benchmark.ipynb
```

Para medir se necesita el entorno levantado con el perfil `high`, que define
`OLLAMA_CORRECTION_MODEL`, y activar la medición en el notebook.

```python
RUN_BENCHMARK = True
FORCE = False
```

Una configuración ya medida se reutiliza. `FORCE = True` reemplaza su resultado,
mientras que aumentar `FLOW_VERSION` añade una medición sin borrar las anteriores.
`results.json` conserva el contrato v2, con la base en primer lugar y
una huella de configuración para reconocer mediciones equivalentes. Las tablas se calculan desde ese registro.

## Referencias

- *Historical Ink: 19th-century Latin American Spanish newspaper corpus with LLM OCR correction* (2024), corrección de prensa histórica en español y análisis de alucinaciones.
- Boros et al. (2024), *Post-correction of historical text transcripts with large language models*, evaluación de ejemplos en el prompt de modelos abiertos.
- Bourne (2024), *CLOCR-C: Context leveraging OCR correction with pre-trained language models*, uso del contexto cercano.
- HIPE-OCRepair (ICDAR 2026), evaluación de sobrecorrección y protección de entidades.